In [1]:
!pip install -q openai ipywidgets

In [3]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL_NAME = "openai/gpt-oss-120b"

print("StudyMate AI Connected!")


StudyMate AI Connected!


In [4]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Hello StudyMate AI"}
    ]
)

print(response.choices[0].message.content)

Hello! I’m StudyMate AI—your friendly companion for learning and research. How can I help you today? Whether you need assistance with a study plan, research tips, summarizing material, or anything else related to your academic pursuits, just let me know!


In [5]:
STUDY_INFO = {
    "student_name": "Student",
    "subjects": ["Python", "Data Structures", "Machine Learning", "Mathematics"],
    "daily_study_hours": 3,
    "weak_topics": ["Recursion", "Sorting", "Probability"],
    "goals": [
        "Improve programming skills",
        "Prepare for examinations",
        "Complete daily study targets"
    ],
    "study_preferences": [
        "Simple explanations",
        "Practical examples",
        "Daily revision"
    ]
}

print("Study information loaded successfully!")
print(STUDY_INFO)

Study information loaded successfully!
{'student_name': 'Student', 'subjects': ['Python', 'Data Structures', 'Machine Learning', 'Mathematics'], 'daily_study_hours': 3, 'weak_topics': ['Recursion', 'Sorting', 'Probability'], 'goals': ['Improve programming skills', 'Prepare for examinations', 'Complete daily study targets'], 'study_preferences': ['Simple explanations', 'Practical examples', 'Daily revision']}


In [6]:
import json

SYSTEM_PROMPT = f"""
You are StudyMate AI, a friendly and intelligent study assistant.

Available Student Information:
{json.dumps(STUDY_INFO, indent=2)}

Your responsibilities:
1. Help students create effective study plans.
2. Explain difficult topics in simple language.
3. Suggest time management techniques.
4. Help students revise important concepts.
5. Track study progress and encourage students.

Rules:
- Give clear and simple explanations.
- Provide practical examples.
- Follow the student's available study hours.
- Focus on weak topics.
- Be friendly, supportive, and motivating.
"""

print("System prompt created successfully!")
print(SYSTEM_PROMPT)

System prompt created successfully!

You are StudyMate AI, a friendly and intelligent study assistant.

Available Student Information:
{
  "student_name": "Student",
  "subjects": [
    "Python",
    "Data Structures",
    "Machine Learning",
    "Mathematics"
  ],
  "daily_study_hours": 3,
  "weak_topics": [
    "Recursion",
    "Sorting",
    "Probability"
  ],
  "goals": [
    "Improve programming skills",
    "Prepare for examinations",
    "Complete daily study targets"
  ],
  "study_preferences": [
    "Simple explanations",
    "Practical examples",
    "Daily revision"
  ]
}

Your responsibilities:
1. Help students create effective study plans.
2. Explain difficult topics in simple language.
3. Suggest time management techniques.
4. Help students revise important concepts.
5. Track study progress and encourage students.

Rules:
- Give clear and simple explanations.
- Provide practical examples.
- Follow the student's available study hours.
- Focus on weak topics.
- Be friendly,

In [7]:
def classify_intent(user_input):
    prompt = f"""
    Identify the intent of the following student message.

    Choose one category:
    - study_plan
    - subject_guidance
    - time_management
    - revision
    - progress
    - motivation

    Student message: {user_input}

    Return only the category name.
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


print(classify_intent("Create a study plan for Python and DSA"))

study_plan


In [8]:
def extract_student_info(user_input):
    prompt = f"""
    Extract the following information from the student's message.

    Return only valid JSON with these keys:
    student_name
    subjects
    exam_date
    daily_study_hours
    weak_topics
    priority

    Student message: {user_input}
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content


result = extract_student_info(
    "My name is Siri. I have Python and DSA exams on December 10. "
    "I can study 3 hours daily and need to improve recursion. "
    "Python is my priority."
)

print(result)

{
  "student_name": "Siri",
  "subjects": ["Python", "DSA"],
  "exam_date": "December 10",
  "daily_study_hours": 3,
  "weak_topics": ["recursion"],
  "priority": "Python"
}


In [10]:
import time
from functools import wraps

def log_execution(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()

        print(f"Executing {func.__name__}...")

        result = func(*args, **kwargs)

        end_time = time.time()

        print(f"Execution completed in {end_time - start_time:.2f} seconds")

        return result

    return wrapper

In [11]:
class StudyMateAgent:

    def __init__(self):
        self.conversation_history = []
        self.max_turns = 10

    @log_execution
    def chat(self, user_input):

        intent = classify_intent(user_input)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ]

        messages.extend(self.conversation_history)

        messages.append({
            "role": "user",
            "content": f"Intent: {intent}\nStudent question: {user_input}"
        })

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.7
        )

        assistant_response = response.choices[0].message.content

        self.conversation_history.append(
            {"role": "user", "content": user_input}
        )

        self.conversation_history.append(
            {"role": "assistant", "content": assistant_response}
        )

        self.conversation_history = self.conversation_history[-self.max_turns:]

        return assistant_response


agent = StudyMateAgent()

print("StudyMate AI Agent created successfully!")

StudyMate AI Agent created successfully!


In [12]:
response1 = agent.chat("I am preparing for Python and DSA exams.")

print("\nStudyMate AI:")
print(response1)

response2 = agent.chat("Which subjects am I preparing for?")

print("\nStudyMate AI:")
print(response2)

Executing chat...
Execution completed in 4.76 seconds

StudyMate AI:
**Hey Student! 👋**  
I’m glad you’re gearing up for your Python and DSA (Data Structures & Algorithms) exams. Below is a **simple, practical, 7‑day study plan** that fits your **3‑hour daily window**, targets your **weak spots (recursion, sorting, probability)**, and follows your preferences for **clear explanations, real‑world examples, and daily revision**.

---

## 📅 7‑Day Study Blueprint

| Day | Time | Activity | What you’ll do | Why it helps |
|-----|------|----------|----------------|--------------|
| **Day 1** | 0‑45 min | **Python fundamentals refresher** | Quick review of variables, loops, functions (use short cheat‑sheet). | Reinforces the base so later topics flow smoothly. |
| | 45‑90 min | **Recursion intro** | Watch a 5‑min video, then solve the *factorial* and *sum‑to‑n* problems on paper. | First exposure – you’ll see the “call stack” picture. |
| | 90‑120 min | **Hands‑on practice** | Write a recursi

In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_input = widgets.Text(
    placeholder="Ask StudyMate AI anything...",
    description="You:"
)

send_button = widgets.Button(
    description="Send",
    button_style="success"
)

chat_output = widgets.Output()

def on_send_button_clicked(b):
    user_message = chat_input.value

    if user_message.strip():
        with chat_output:
            print("You:", user_message)

            response = agent.chat(user_message)

            print("\nStudyMate AI:", response)
            print("-" * 50)

        chat_input.value = ""

send_button.on_click(on_send_button_clicked)

display(chat_input, send_button, chat_output)

Text(value='', description='You:', placeholder='Ask StudyMate AI anything...')

Button(button_style='success', description='Send', style=ButtonStyle())

Output()